In [ ]:
"""
================================================================================
Trailing Stop-Loss (CAPPED) — OOS Full-Year Simulation (Step8 + Step9)
Using PREBUILT 15-Minute Parquet Bars — Next-Day Only + Initial SL >= 1%
================================================================================

SUMMARY
-------
Runs your trailing stop-loss overlay on trade-level data using prebuilt 15-minute
OHLC parquet files (one per symbol).

Core rules (same methodology as earlier):
1) Initial SL placed at entry close, min 1% (INITIAL_MIN_STOP_PCT), capped by MAX_STOP_*
2) Monitoring starts NEXT DAY only (no same-day trailing)
3) Adaptive trailing distance based on quadratic slope regime vs ATR fallback
4) Exit reasons: InitialSL, TrailingSL, OriginalExit, etc.
5) Outputs full trade-level results + summary + debug reason counts

IMPORTANT: TIMEZONE NORMALIZATION
--------------------------------
Your rs15 parquet _dt is often tz-aware UTC. Trade CSV timestamps are usually tz-naive.
This script converts rs15 _dt to tz-naive UTC so comparisons work reliably.

================================================================================
"""

In [ ]:

TRADE_CSV_PATH = "<local-path>"

In [ ]:
import os
import numpy as np
import pandas as pd
from dataclasses import dataclass
from datetime import timedelta

# ============================================================
# CONFIG (EDIT THESE PATHS)
# ============================================================

TRADE_CSV_PATH = "<local-path>"
MAPPING_XLSX_PATH = "<local-path>"

# Prebuilt 15-min parquet root
RS15_ROOT = "<local-path>"
RS15_FILE_NAME = "rs_15min.parquet"   # or "rs_15T.parquet" if you used that name

# Step 8 base params (dir-specific)
STEP8_DIR = "<local-path>"
BEST_STEP8_PATH = os.path.join(STEP8_DIR, "best_config_step8.csv")

# Step 9 sensitivity
STEP9_DIR = "<local-path>"
BEST_STEP9_PATH = os.path.join(STEP9_DIR, "best_config_step9.csv")

# ============================================================
# DATE FILTER (UPDATED)
# ============================================================
START_DATE = pd.Timestamp("2025-01-01")   # inclusive
END_DATE   = None                        # e.g. pd.Timestamp("2026-04-30") or None for "till latest available"

OUT_DIR = os.path.join(
    os.path.dirname(TRADE_CSV_PATH),
    f"TrailingSL_OOS_FROM_{START_DATE.strftime('%Y%m%d')}"
    + (f"_TO_{END_DATE.strftime('%Y%m%d')}" if END_DATE is not None else "_TO_LATEST")
    + "_FULLYEAR_STEP8STEP9_NEXTDAY_ONLY_INITIALSL_USING_RS15"
)
os.makedirs(OUT_DIR, exist_ok=True)

# ============================================================
# Locked settings (same as pilot)
# ============================================================

ATR_PERIOD = 14
REG_WINDOW = 16

TRIGGER_ON_CLOSE = True
ATR_FALLBACK_MULT = 3.0

# --- Initial SL must be >= 1% ---
INITIAL_MIN_STOP_PCT = 0.01   # 1%
# trailing minimum (can be tighter)
MIN_STOP_PCT = 0.0010         # 0.10%

# Rule: apply trailing/monitoring from next day only
APPLY_FROM_NEXT_DAY_ONLY = True

PRINT_EVERY = 500

# ============================================================
# Robust CSV reader
# ============================================================
def read_trade_csv(path: str) -> pd.DataFrame:
    for enc in ["utf-8", "cp1252", "latin1"]:
        try:
            return pd.read_csv(path, encoding=enc, low_memory=False)
        except UnicodeDecodeError:
            continue
    return pd.read_csv(path, encoding="utf-8", encoding_errors="ignore", low_memory=False)

# ============================================================
# Helpers
# ============================================================
def normalize_ticker(x: str) -> str:
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return ""
    x = str(x).strip().upper()
    return "".join([ch for ch in x if ch.isalnum()])

def trade_base_from_currency(currency: str) -> str:
    if currency is None or (isinstance(currency, float) and np.isnan(currency)):
        return ""
    s = str(currency).strip().upper()
    base = s.split("/")[0].strip() if "/" in s else s
    return normalize_ticker(base)

def normalize_direction(x) -> str:
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return "UNKNOWN"
    s = str(x).strip().upper()
    if s in ["LONG", "BUY", "B", "1", "1.0", "+1", "+1.0"]:
        return "LONG"
    if s in ["SHORT", "SELL", "S", "-1", "-1.0"]:
        return "SHORT"
    try:
        v = float(s)
        if v > 0: return "LONG"
        if v < 0: return "SHORT"
    except Exception:
        pass
    return "UNKNOWN"

def tz_naive_utc(s: pd.Series) -> pd.Series:
    """
    Convert datetime series to tz-naive UTC.
    If tz-aware -> tz_convert('UTC') then tz_localize(None).
    If tz-naive -> unchanged.
    """
    if not pd.api.types.is_datetime64_any_dtype(s):
        s = pd.to_datetime(s, errors="coerce")
    try:
        tz = getattr(s.dt, "tz", None)
    except Exception:
        tz = None
    if tz is not None:
        s = s.dt.tz_convert("UTC").dt.tz_localize(None)
    return s

# ============================================================
# Load mapping file (TradeBase -> rs15 ticker folder)
# ============================================================
def load_trade_to_rs15_mapping(mapping_xlsx_path: str) -> dict:
    mp = pd.read_excel(mapping_xlsx_path)
    mp.columns = [c.strip() for c in mp.columns]
    if "org_symbol" not in mp.columns or "Symbol" not in mp.columns:
        raise ValueError("Mapping file must contain columns: org_symbol, Symbol")

    mp["rs15_ticker"] = mp["org_symbol"].apply(normalize_ticker)
    mp["trade_base"] = (
        mp["Symbol"].astype(str).str.upper().str.strip().str.split("/").str[0].apply(normalize_ticker)
    )

    mp = mp[(mp["trade_base"] != "") & (mp["rs15_ticker"] != "")]
    mp = mp.drop_duplicates(subset=["trade_base"], keep="first")
    return dict(zip(mp["trade_base"], mp["rs15_ticker"]))

TRADE_TO_RS15 = load_trade_to_rs15_mapping(MAPPING_XLSX_PATH)

def map_trade_base_to_rs15(trade_base: str) -> str:
    t = normalize_ticker(trade_base)
    return TRADE_TO_RS15.get(t, t)

# ============================================================
# Discover available rs15 tickers (for validation)
# ============================================================
def load_rs15_tickers(rs15_root: str) -> set:
    out = set()
    if not os.path.exists(rs15_root):
        return out
    for name in os.listdir(rs15_root):
        if name.lower().startswith("symbol="):
            sym = name.split("=", 1)[1].strip()
            out.add(normalize_ticker(sym))
    return out

RS15_TICKERS = load_rs15_tickers(RS15_ROOT)

# ============================================================
# ATR
# ============================================================
def compute_atr(df: pd.DataFrame, period: int) -> pd.Series:
    high = df["high"]
    low = df["low"]
    close = df["close"]
    tr = pd.concat([
        high - low,
        (high - close.shift(1)).abs(),
        (low - close.shift(1)).abs()
    ], axis=1).max(axis=1)
    return tr.rolling(period, min_periods=period).mean()

# ============================================================
# Load RS15 parquet per ticker + ATR (cached) — TZ FIX HERE
# ============================================================
_RS_CACHE = {}

def prepare_rs_for_ticker(ticker: str):
    """
    Loads prebuilt 15-min OHLC parquet for ticker, computes ATR(14),
    and returns df: _dt (tz-naive UTC), open, high, low, close, ATR
    """
    t = normalize_ticker(ticker)
    if t in _RS_CACHE:
        return _RS_CACHE[t]

    fp = os.path.join(RS15_ROOT, f"symbol={t}", RS15_FILE_NAME)
    if not os.path.exists(fp):
        _RS_CACHE[t] = None
        return None

    rs = pd.read_parquet(fp)
    if rs is None or rs.empty:
        _RS_CACHE[t] = None
        return None

    rs.columns = [c.strip() for c in rs.columns]
    need = ["_dt", "open", "high", "low", "close"]
    if not all(c in rs.columns for c in need):
        _RS_CACHE[t] = None
        return None

    rs["_dt"] = tz_naive_utc(rs["_dt"])
    rs = rs.dropna(subset=["_dt"]).sort_values("_dt").reset_index(drop=True)

    for c in ["open", "high", "low", "close"]:
        rs[c] = pd.to_numeric(rs[c], errors="coerce")
    rs = rs.dropna(subset=["open", "high", "low", "close"])

    if rs.empty:
        _RS_CACHE[t] = None
        return None

    rs["ATR"] = compute_atr(rs, ATR_PERIOD)

    _RS_CACHE[t] = rs
    return rs

# ============================================================
# Load & merge config (Step8 base + Step9 sensitivity)
# ============================================================
def load_best_step8(path: str) -> dict:
    b8 = pd.read_csv(path).iloc[0].to_dict()
    need = ["ATAN_LONG","ATAN_SHORT","MAX_STOP_LONG","MAX_STOP_SHORT"]
    miss = [k for k in need if k not in b8 or pd.isna(b8.get(k))]
    if miss:
        raise ValueError(f"best_config_step8.csv missing/NaN keys: {miss}")
    return {k: float(b8[k]) for k in need}

def load_best_step9(path: str) -> dict:
    b9 = pd.read_csv(path).iloc[0].to_dict()
    return {
        "SENS_LONG": float(b9.get("SENS_LONG", 1.0)),
        "SENS_SHORT": float(b9.get("SENS_SHORT", 1.0)),
    }

def load_merged_config() -> dict:
    base = load_best_step8(BEST_STEP8_PATH)
    sens = load_best_step9(BEST_STEP9_PATH)
    cfg = {**base, **sens}
    print("Merged config:", cfg)
    return cfg

# ============================================================
# Trade extraction (UPDATED FILTER)
# ============================================================
def build_trade_summary(raw: pd.DataFrame) -> pd.DataFrame:
    df = raw.copy()
    df.columns = [c.strip() for c in df.columns]

    required = ["TradeID","Currency","Direction","date","price","Opening Date","Closing Date"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Trade file missing required columns: {missing}")

    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df["Opening Date"] = pd.to_datetime(df["Opening Date"], errors="coerce")
    df["Closing Date"] = pd.to_datetime(df["Closing Date"], errors="coerce")

    for c in ["date", "Opening Date", "Closing Date"]:
        df[c] = tz_naive_utc(df[c])

    df["price"] = pd.to_numeric(df["price"], errors="coerce")

    # ---- NEW: keep everything from START_DATE onward (and optional END_DATE)
    df = df[df["date"].notna()].copy()
    df = df[df["date"] >= START_DATE].copy()
    if END_DATE is not None:
        df = df[df["date"] <= END_DATE].copy()

    df["TradeBase"] = df["Currency"].apply(trade_base_from_currency)
    df["TickerNorm"] = df["TradeBase"].apply(map_trade_base_to_rs15)
    df["DirectionNorm"] = df["Direction"].apply(normalize_direction)

    rows = []
    for tid, g in df.groupby("TradeID", dropna=True):
        g = g.sort_values("date")

        entry_date = g["Opening Date"].dropna().iloc[0] if g["Opening Date"].notna().any() else pd.NaT
        exit_date  = g["Closing Date"].dropna().iloc[0] if g["Closing Date"].notna().any() else pd.NaT

        direction = g["DirectionNorm"].dropna().iloc[0] if g["DirectionNorm"].notna().any() else "UNKNOWN"
        ticker = g["TickerNorm"].dropna().iloc[0] if g["TickerNorm"].notna().any() else ""

        entry_px = np.nan
        if pd.notna(entry_date):
            day_rows = g[g["date"].dt.normalize() == entry_date.normalize()]
            if day_rows["price"].notna().any():
                entry_px = float(day_rows["price"].dropna().iloc[-1])

        exit_px = np.nan
        if pd.notna(exit_date):
            day_rows = g[g["date"].dt.normalize() == exit_date.normalize()]
            if day_rows["price"].notna().any():
                exit_px = float(day_rows["price"].dropna().iloc[-1])

        valid_window = pd.notna(entry_date) and pd.notna(exit_date) and (exit_date.normalize() > entry_date.normalize())
        core_ok = (
            ticker != "" and ticker in RS15_TICKERS and
            direction != "UNKNOWN" and
            pd.notna(entry_date) and pd.notna(exit_date) and
            pd.notna(entry_px) and pd.notna(exit_px)
        )

        rows.append({
            "TradeID": tid,
            "TickerNorm": ticker,
            "Direction": direction,
            "EntryDate": entry_date,
            "ExitDate_Original": exit_date,
            "EntryPrice": entry_px,
            "ExitPrice_Original": exit_px,
            "HasAllCoreFields": bool(core_ok),
            "HasValidTrailingWindow": bool(valid_window)
        })
    return pd.DataFrame(rows)

# ============================================================
# Trailing simulation (UNCHANGED)
# ============================================================
def adaptive_stop_distance_pct(bars: pd.DataFrame,
                               atan_threshold: float,
                               atr_fallback_mult: float,
                               min_stop_pct: float,
                               max_stop_pct: float,
                               sensitivity: float) -> float:
    if len(bars) < REG_WINDOW or bars["ATR"].dropna().empty:
        return np.nan

    recent = bars["close"].iloc[-REG_WINDOW:].astype(float)
    if len(recent) < REG_WINDOW:
        return np.nan

    x = np.arange(len(recent), dtype=float)
    a, b, _c = np.polyfit(x, recent.values, 2)

    slope_norm = (2 * a * x[-1] + b) / recent.iloc[-1]
    accel = (2 * a)
    atan_slope = float(np.arctan(slope_norm))

    atr = bars["ATR"].iloc[-1]
    px = bars["close"].iloc[-1]
    atr_pct = float(atr / px) if (pd.notna(atr) and atr > 0 and px > 0) else np.nan
    if pd.isna(atr_pct):
        return np.nan

    if abs(atan_slope) > atan_threshold:
        stop_pct = abs(slope_norm) * sensitivity * (1.0 + abs(accel))
        return float(np.clip(stop_pct, min_stop_pct, max_stop_pct))

    stop_pct = atr_pct * atr_fallback_mult
    return float(np.clip(stop_pct, min_stop_pct, max_stop_pct))

def compute_initial_stop_pct(rs: pd.DataFrame,
                             entry_date: pd.Timestamp,
                             entry_px: float,
                             direction: str,
                             cfg: dict) -> float:
    direction = normalize_direction(direction)
    is_long = (direction == "LONG")

    atan_thr = cfg["ATAN_LONG"] if is_long else cfg["ATAN_SHORT"]
    max_stop = cfg["MAX_STOP_LONG"] if is_long else cfg["MAX_STOP_SHORT"]
    sens     = cfg["SENS_LONG"] if is_long else cfg["SENS_SHORT"]

    entry_close_dt = entry_date.normalize() + pd.Timedelta(hours=23, minutes=59, seconds=59)
    hist = rs[rs["_dt"] <= entry_close_dt].copy()

    if hist.empty:
        return float(np.clip(INITIAL_MIN_STOP_PCT, INITIAL_MIN_STOP_PCT, max_stop))

    stop_pct = adaptive_stop_distance_pct(
        hist, atan_thr, ATR_FALLBACK_MULT, INITIAL_MIN_STOP_PCT, max_stop, sens
    )
    if pd.notna(stop_pct):
        return float(stop_pct)

    last = hist.iloc[-1]
    atr = last.get("ATR", np.nan)
    px  = last.get("close", np.nan)
    if pd.notna(atr) and atr > 0 and pd.notna(px) and px > 0:
        atr_pct = float(atr / px)
        stop_pct = atr_pct * ATR_FALLBACK_MULT
        return float(np.clip(stop_pct, INITIAL_MIN_STOP_PCT, max_stop))

    return float(np.clip(INITIAL_MIN_STOP_PCT, INITIAL_MIN_STOP_PCT, max_stop))

@dataclass
class TrailResult:
    exit_time: pd.Timestamp
    exit_price: float
    exit_reason: str
    initial_stop_pct: float
    initial_stop_price: float

def price_improvement(direction: str, new_exit: float, old_exit: float) -> float:
    d = normalize_direction(direction)
    return (new_exit - old_exit) if d == "LONG" else (old_exit - new_exit)

def simulate_trade(tr: pd.Series, rs: pd.DataFrame, cfg: dict) -> TrailResult:
    entry_date = pd.to_datetime(tr["EntryDate"])
    exit_date  = pd.to_datetime(tr["ExitDate_Original"])
    entry_px   = float(tr["EntryPrice"])
    orig_exit_px = float(tr["ExitPrice_Original"])

    if exit_date.normalize() <= entry_date.normalize():
        return TrailResult(exit_date, orig_exit_px, "SameDayOrInvalidWindow", np.nan, np.nan)

    direction = normalize_direction(tr["Direction"])
    is_long = (direction == "LONG")

    init_stop_pct = compute_initial_stop_pct(rs, entry_date, entry_px, direction, cfg)
    if is_long:
        stop = entry_px * (1.0 - init_stop_pct)
    else:
        stop = entry_px * (1.0 + init_stop_pct)

    initial_stop_price = float(stop)
    best_close = entry_px
    stop_source = "Initial"

    start_dt = entry_date.normalize() + timedelta(days=1)
    end_dt   = exit_date + timedelta(days=1)

    w = rs[(rs["_dt"] >= start_dt) & (rs["_dt"] < end_dt)].copy()
    if w.empty:
        return TrailResult(exit_date, orig_exit_px, "NoRSDataInWindow",
                           float(init_stop_pct), float(initial_stop_price))

    for i in range(len(w)):
        sub = w.iloc[:i+1]
        row = w.iloc[i]
        dt = row["_dt"]

        close = float(row["close"])
        high = float(row["high"])
        low  = float(row["low"])

        if is_long:
            best_close = max(best_close, close)
        else:
            best_close = min(best_close, close)

        if is_long:
            hit_val = close if TRIGGER_ON_CLOSE else low
            if hit_val <= stop:
                reason = "InitialSL" if stop_source == "Initial" else "TrailingSL"
                return TrailResult(dt, float(stop), reason,
                                   float(init_stop_pct), float(initial_stop_price))
        else:
            hit_val = close if TRIGGER_ON_CLOSE else high
            if hit_val >= stop:
                reason = "InitialSL" if stop_source == "Initial" else "TrailingSL"
                return TrailResult(dt, float(stop), reason,
                                   float(init_stop_pct), float(initial_stop_price))

        atan_thr = cfg["ATAN_LONG"] if is_long else cfg["ATAN_SHORT"]
        max_stop = cfg["MAX_STOP_LONG"] if is_long else cfg["MAX_STOP_SHORT"]
        sens     = cfg["SENS_LONG"] if is_long else cfg["SENS_SHORT"]

        stop_pct = adaptive_stop_distance_pct(
            sub, atan_thr, ATR_FALLBACK_MULT, MIN_STOP_PCT, max_stop, sens
        )
        if pd.isna(stop_pct):
            continue

        if is_long:
            stop = max(stop, best_close * (1.0 - stop_pct))
        else:
            stop = min(stop, best_close * (1.0 + stop_pct))

        stop_source = "Trailing"

    return TrailResult(exit_date, orig_exit_px, "OriginalExit",
                       float(init_stop_pct), float(initial_stop_price))

def summarize(out: pd.DataFrame) -> pd.DataFrame:
    df = out.copy()
    df["TriggeredFlag"] = df["ExitReason"].isin(["TrailingSL", "InitialSL"])
    df["ImprovedFlag"] = df["PriceImprovement"] > 0

    def agg(g):
        n = len(g)
        trig = int(g["TriggeredFlag"].sum())
        return {
            "Trades": n,
            "SLTriggered": trig,
            "PctTriggered": trig / n if n else np.nan,
            "Triggered_PctImproved": float(g.loc[g["TriggeredFlag"], "ImprovedFlag"].mean()) if trig else np.nan,
            "AvgPctImpact_All": float(g["PctImprovement_vs_OriginalExit"].mean(skipna=True)),
            "AvgPctImpact_Triggered": float(g.loc[g["TriggeredFlag"], "PctImprovement_vs_OriginalExit"].mean(skipna=True)) if trig else np.nan,
        }

    rows = [{"Group":"ALL", **agg(df)}]
    for d, g in df.groupby(df["Direction"].astype(str).str.upper().str.strip()):
        rows.append({"Group": f"Direction={d}", **agg(g)})
    return pd.DataFrame(rows)

# ============================================================
# MAIN
# ============================================================
def main():
    cfg = load_merged_config()

    raw = read_trade_csv(TRADE_CSV_PATH)
    trades = build_trade_summary(raw)

    print("\nDate filter:",
          f"FROM {START_DATE.date()}",
          f"TO {END_DATE.date()}" if END_DATE is not None else "TO LATEST")

    print("\nTradeIDs in range (pre-filter):", trades["TradeID"].nunique())
    print("Trades with core fields OK:", int(trades["HasAllCoreFields"].sum()))
    print("Trades with valid trailing window:", int(trades["HasValidTrailingWindow"].sum()))

    eligible = trades[(trades["HasAllCoreFields"]) & (trades["HasValidTrailingWindow"])].copy()
    if eligible.empty:
        raise RuntimeError("No eligible trades after filtering (core fields + exit>entry).")

    eligible = eligible.sort_values(["TickerNorm", "EntryDate", "TradeID"]).reset_index(drop=True)
    print("\nEligible trades to process:", len(eligible))
    print("Unique tickers in eligible:", eligible["TickerNorm"].nunique())

    tickers = sorted(eligible["TickerNorm"].unique())
    print("\nPreloading rs15 data for tickers:", len(tickers))
    for i, tkr in enumerate(tickers, 1):
        prepare_rs_for_ticker(tkr)
        if i % 25 == 0:
            print(f"  Loaded {i}/{len(tickers)} tickers...")

    out_rows = []
    dbg_rows = []

    for i, tr in eligible.iterrows():
        rs = prepare_rs_for_ticker(tr["TickerNorm"])
        if rs is None or rs.empty:
            dbg_rows.append({"TradeID": tr["TradeID"], "Reason": "MissingRSData"})
            continue

        res = simulate_trade(tr, rs, cfg)

        old_exit = float(tr["ExitPrice_Original"])
        new_exit = float(res.exit_price)
        impr = price_improvement(tr["Direction"], new_exit, old_exit)

        out_rows.append({
            "TradeID": tr["TradeID"],
            "TickerNorm": tr["TickerNorm"],
            "Direction": tr["Direction"],
            "EntryDate": tr["EntryDate"],
            "ExitDate_Original": tr["ExitDate_Original"],
            "EntryPrice": float(tr["EntryPrice"]),
            "InitialStopPct": res.initial_stop_pct,
            "InitialStopPrice": res.initial_stop_price,
            "ExitPrice_Original": old_exit,
            "ExitTime_Trailing": res.exit_time,
            "ExitPrice_Trailing": new_exit,
            "ExitReason": res.exit_reason,
            "PriceImprovement": impr,
            "PctImprovement_vs_OriginalExit": (impr / old_exit) if old_exit else np.nan
        })
        dbg_rows.append({"TradeID": tr["TradeID"], "Reason": res.exit_reason})

        if (i + 1) % PRINT_EVERY == 0:
            print(f"Processed {i+1}/{len(eligible)} trades...")

    out_df = pd.DataFrame(out_rows)
    summ_df = summarize(out_df)
    dbg_df = pd.DataFrame(dbg_rows)

    reason_counts = dbg_df["Reason"].value_counts(dropna=False)

    out_path = os.path.join(OUT_DIR, "trades_full_from_20250101.csv")
    summ_path = os.path.join(OUT_DIR, "summary_full_from_20250101.csv")
    dbg_path = os.path.join(OUT_DIR, "debug_reasons_full_from_20250101.csv")
    reason_path = os.path.join(OUT_DIR, "reason_counts_full_from_20250101.csv")

    out_df.to_csv(out_path, index=False)
    summ_df.to_csv(summ_path, index=False)
    dbg_df.to_csv(dbg_path, index=False)
    reason_counts.rename_axis("Reason").reset_index(name="Count").to_csv(reason_path, index=False)

    print("\nSaved:", out_path)
    print("Saved:", summ_path)
    print("Saved:", dbg_path)
    print("Saved:", reason_path)

    print("\n=== FULL SUMMARY ===")
    print(summ_df.to_string(index=False))

    print("\n=== EXIT REASONS ===")
    print(reason_counts.to_string())

if __name__ == "__main__":
    main()

In [ ]:
# Recreate trades object exactly like in main()

raw = read_trade_csv(TRADE_CSV_PATH)
trades = build_trade_summary(raw)

print("Total TradeIDs:", trades["TradeID"].nunique())
print("Core OK:", trades["HasAllCoreFields"].sum())
print("Valid Window:", trades["HasValidTrailingWindow"].sum())

In [ ]:
def core_fail_breakdown(trades, rs15_tickers):
    rs15_tickers = set(rs15_tickers)

    conds = {
        "MissingTicker": trades["TickerNorm"].astype(str).str.strip().eq(""),
        "TickerNotInRS15": ~trades["TickerNorm"].astype(str).isin(rs15_tickers),
        "UnknownDirection": trades["Direction"].astype(str).str.upper().eq("UNKNOWN"),
        "MissingEntryDate": trades["EntryDate"].isna(),
        "MissingExitDate": trades["ExitDate_Original"].isna(),
        "MissingEntryPrice": trades["EntryPrice"].isna(),
        "MissingExitPrice": trades["ExitPrice_Original"].isna(),
    }

    out = []
    for k, m in conds.items():
        out.append((k, int(m.sum())))

    out_df = pd.DataFrame(out, columns=["Reason", "Count"]).sort_values("Count", ascending=False)
    return out_df

print(core_fail_breakdown(trades, RS15_TICKERS).to_string(index=False))